In [ ]:
import h5py
import pandas as pd
import anndata as ad
import numpy as np
import pickle
import cooler
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib import cm as cm
import re
import xarray as xr
from scipy.stats import norm
from scipy.signal import find_peaks
from scipy.stats import pearsonr, zscore
import seaborn as sns

In [ ]:
with open("/scratch/mmoore/Epitherapy3D/analysis/HiC/paper_final/5_snm3c/higashi/50kb/label_info.pickle", "rb") as f:
    label_info = pickle.load(f)

In [ ]:
meta = pd.read_csv('../schicluster/CellMetadata.PassQC.Quintile.csv.gz', sep=',', header=0, index_col=0)
meta

In [ ]:
f = h5py.File('10kb/temp_redo_dir/GBM.hdf5', 'r')

In [ ]:
insulation = f['insulation']

In [ ]:
genome_scTAD = []
with h5py.File("10kb/temp_redo_dir/GBM.hdf5", "r") as cp_f:
    print(cp_f.keys())
    cp = cp_f['tads']

    for id_ in range(7467):
        v = np.array(cp['cell_%d' % id_])
        genome_scTAD.append(v[:])

genome_scTAD = np.stack(genome_scTAD, axis=0)
print(genome_scTAD.shape)

In [ ]:
x = np.array(insulation['bin']['chrom'])
v = np.array(insulation['bin']['start'])

In [ ]:
# Build column names from chromosome + start position
col_names = [x[i].decode() + "_" + str(v[i]) for i in range(len(v))]

# Create DataFrame
df = pd.DataFrame(genome_scTAD, columns=col_names)

print(df.shape)
df.head()

In [ ]:
bound_count_ct = df.groupby(meta["qcluster"].values).sum()

In [ ]:
indir = '../snm3c_all/new/schicluster/'

In [ ]:
leg = ['B36_0', 'B36_1', 'B36_2', 'B36_3', 'B36_4', 'B36_5', 'B36_6', 'B49_0', 'B49_1', 'B49_2', 'B49_3', 'B49_4', 'B49_5', 'B49_6', 'B66_0', 'B66_1', 'B66_2', 'B66_3', 'B66_4', 'B66_5', 'B66_6', 'NHA_0', 'NHA_1', 'NHA_2', 'NHA_3', 'NHA_4', 'NHA_5', 'NHA_6', 'qNHA_0', 'qNHA_1', 'qNHA_2', 'qNHA_3', 'qNHA_4', 'qNHA_5', 'qNHA_6'
               ]
legname = ['B36_0', 'B36_1', 'B36_2', 'B36_3', 'B36_4', 'B36_5', 'B36_6', 'B49_0', 'B49_1', 'B49_2', 'B49_3', 'B49_4', 'B49_5', 'B49_6', 'B66_0', 'B66_1', 'B66_2', 'B66_3', 'B66_4', 'B66_5', 'B66_6', 'NHA_0', 'NHA_1', 'NHA_2', 'NHA_3', 'NHA_4', 'NHA_5', 'NHA_6', 'qNHA_0', 'qNHA_1', 'qNHA_2', 'qNHA_3', 'qNHA_4', 'qNHA_5', 'qNHA_6'
                   ]
leg2name = {xx:yy for xx,yy in zip(leg, legname)}

In [ ]:
leg = {'B36': ['B36_0', 'B36_1', 'B36_2', 'B36_3', 'B36_4', 'B36_5', 'B36_6'], 
       'B49': ['B49_0', 'B49_1', 'B49_2', 'B49_3', 'B49_4', 'B49_5', 'B49_6'], 
       'B66': ['B66_0', 'B66_1', 'B66_2', 'B66_3', 'B66_4', 'B66_5', 'B66_6'], 
       'NHA': ['NHA_0', 'NHA_1', 'NHA_2', 'NHA_3', 'NHA_4', 'NHA_5', 'NHA_6'], 
       'qNHA': ['qNHA_0', 'qNHA_1', 'qNHA_2', 'qNHA_3', 'qNHA_4', 'qNHA_5', 'qNHA_6'], 
}
leg['all'] = leg['B36'] + leg['B49'] + leg['B66'] + leg['NHA'] + leg['qNHA']
leg['GSC'] = leg['B36'] + leg['B49'] + leg['B66']

In [ ]:
group_name = 'GSC'

In [ ]:
leg = pd.Index(leg[group_name])
legname = leg.map(leg2name)
res = 10000

In [ ]:
bound_count_ct = bound_count_ct.loc[leg]

In [ ]:
cell_count_ct = pd.read_csv(f'{indir}MajorType_cellcount.csv.gz', index_col=0, header=0, squeeze=True).loc[leg]
bound_prob_ct = (bound_count_ct / cell_count_ct[:,None]).T

In [ ]:
from statsmodels.sandbox.stats.multicomp import multipletests as FDR
from scipy.stats import chi2_contingency

def diff_bound(bound_count_ct, cell_count_ct):
    tmp = cell_count_ct[:,None] - bound_count_ct
    stats = np.zeros(bound_count_ct.shape[1])
    pv = np.ones(bound_count_ct.shape[1])
    binfilter = np.logical_and(bound_count_ct.sum(axis=0)>0, tmp.sum(axis=0)>0)
    for i in range(bound_count_ct.shape[1]):
        if binfilter[i]:
            contig = [bound_count_ct[:,i], tmp[:,i]]
            stats[i], pv[i], _, _ = chi2_contingency(contig)
    fdr = FDR(pv, 0.01, 'fdr_bh')[1]
    return stats, fdr

def shuffle_ct(i):
    global cell_count_ct, sc_border, leg
    np.random.seed(i)
    label = np.random.permutation(sc_border.obs[f'{ct_key}'])
    bound_count_ct = np.array([sc_border.X[label==xx].getnnz(axis=0) for xx in leg])
    bound_prob_ct = bound_count_ct / cell_count_ct[:,None]
    return diff_bound(bound_count_ct, cell_count_ct)[0]

def diff_bound_bulk(ins_count):
    stats = np.zeros(ins_count.shape[2])
    pv = np.ones(ins_count.shape[2])
    binfilter = (ins_count.min(axis=(0,1))>0)
    for i in range(ins_count.shape[2]):
        if binfilter[i]:
            stats[i], pv[i], _, _ = chi2_contingency(ins_count[:,:,i])
    fdr = FDR(pv, 0.01, 'fdr_bh')[1]
    return stats, fdr

In [ ]:
split = df.columns.str.split("_", n=1)
binall = pd.DataFrame({
    "chrom": split.str[0],
    "start": split.str[1].astype(int),
    "end":   split.str[1].astype(int) + 10000
}, index=df.columns)

In [ ]:
binall

In [ ]:
chrom_size_path = '/scratch/devtools/mmoore/genomes/snm3c/hg38/hg38.main.chrom.sort.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24]]
chrom_sizes

In [ ]:
bkl = pd.read_csv(f'{indir}hg38.main.rowsum1000.10kb.bed', sep='\t', header=None, index_col=None)
binall['bklfilter'] = True
for c in chrom_sizes.index:
    chrfilter = (binall['chrom']==c)
    tmp = binall.loc[chrfilter.values]
    tmp.iloc[:10, -1] = False
    tmp.iloc[-10:, -1] = False
    for xx,yy in bkl.loc[bkl[0]==c, [1,2]].values // res:
        tmp.iloc[max([0,xx-2]):(yy+2), -1] = False
    binall.loc[chrfilter] = tmp.copy()

print(binall['bklfilter'].sum())

In [ ]:
chi2sc, fdr_sc = diff_bound(bound_count_ct.values, cell_count_ct.values)
ave = np.mean(chi2sc[chi2sc>0])
stdev = np.std(chi2sc[chi2sc>0])
binall['chi2filter'] = (((chi2sc - ave) / stdev)>norm.isf(0.025))

In [ ]:
print(binall['chi2filter'].sum())

In [ ]:
binall['probdiff'] = (bound_prob_ct.max(axis=1) - bound_prob_ct.min(axis=1)).values
binall['chi2_sc'] = chi2sc.copy()

In [ ]:
sel = []
thres = np.min(chi2sc[fdr_sc<1e-3])
for c in chrom_sizes.index:
    idx = np.where(binall['chrom']==c)[0]
    if len(idx)>0:
        data = chi2sc[idx]
        peaks, _ = find_peaks(data, height=thres, distance=5)
        sel.append(idx.min() + peaks)
        
sel = np.concatenate(sel)

binall['diff_sc'] = 0
binall.loc[binall.index[sel], 'diff_sc'] = 1
binall.loc[:, binall.dtypes=='category'] = binall.loc[:, binall.dtypes=='category'].astype(str)
#binall.to_hdf(f'{outdir}bin_stats.hdf', key='data')

In [ ]:
print(binall['diff_sc'].sum())

In [ ]:
binall

In [ ]:
print((binall['chi2filter'] 
       & binall['diff_sc'] 
       & binall['bklfilter'] 
       & (binall['probdiff']>0.05) 
       #& (binall['insfc']>1.2)
      ).sum())

In [ ]:
selb = (binall['chi2filter'] & binall['diff_sc'] & binall['bklfilter'] & (binall['probdiff']>0.05))
print(sum(selb))

In [ ]:
tmp3c = bound_prob_ct.loc[selb].values
tmp3c = zscore(tmp3c, axis=1)

In [ ]:
cg = sns.clustermap(tmp3c, cmap='bwr', vmin=-3, vmax=3, metric='cosine', col_cluster=False, xticklabels=leg, yticklabels=[], figsize=(6,6))

In [ ]:
cg = sns.clustermap(tmp3c, cmap='bwr', vmin=-3, vmax=3, metric='cosine', 
                    col_cluster=False, xticklabels=leg, yticklabels=[], figsize=(6,6), dendrogram_ratio=(0.05, 0.2))

cg.cax.set_visible(False)

heatmap_pos = cg.ax_heatmap.get_position()
cax = cg.fig.add_axes([heatmap_pos.x1 + 0.02, heatmap_pos.y0 + heatmap_pos.height * 0.3, 0.02, heatmap_pos.height * 0.4])
cb = cg.fig.colorbar(cg.ax_heatmap.collections[0], cax=cax, orientation='vertical')
cax.set_title("Boundary\nProbability\nZ-score", fontsize=9, pad=6, loc='left', x=0)

In [ ]:
binall[selb]

In [ ]:
binall[selb].iloc[:, :3].to_csv("sc_diff_bound.bed", sep="\t", header=False, index=False)